In [0]:
from pyspark.sql.functions import col, to_timestamp, when, expr, current_timestamp, udf, StringType

In [0]:
tabela = spark.read.table("`ct-esteira-dados-aviacao`.trusted.fat_voos_trusted_dep_arr")
tabela_siglas = spark.read.table("`ct-esteira-dados-aviacao`.trusted.dim_aero_siglas_truested")

In [0]:
tabela.createOrReplaceTempView("voos")
tabela_siglas.createOrReplaceTempView("aero_siglas")

In [0]:
tabela_siglas.printSchema()

In [0]:
fat_voos_business_antecip_atrz = spark.sql("""
select 
    v.referencia as data_voo,
    v.sigla_icao_empresa_aerea,
    v.empresa_aerea,
    v.numero_voo,
    ao.s_aero_iata as iata_origem,
    ad.s_aero_iata as iata_destino,
    v.partida_prevista,
    v.partida_real,
    ((unix_timestamp(v.partida_prevista) - unix_timestamp(v.partida_real)) / 60) as antec_dep,
    v.chegada_prevista,
    v.chegada_real,
    ((unix_timestamp(v.chegada_real) - unix_timestamp(v.chegada_prevista)) / 60) as antec_arr
from voos v
INNER JOIN aero_siglas ao ON 
    v.sigla_icao_aeroporto_origem = ao.s_aero_icao
INNER JOIN aero_siglas ad ON
    v.sigla_icao_aeroporto_destino = ad.s_aero_icao
where 
    ao.s_aero_iata <> '---' and
    ad.s_aero_iata <> '---' and
    partida_prevista is not null and
    partida_real is not null and
    chegada_prevista is not null and
    chegada_real is not null and
(
empresa_aerea like 'GOL' or
empresa_aerea like 'LATAM%' or
empresa_aerea like 'AZUL%'
)
order by iata_origem, iata_destino

""")

In [0]:
fat_voos_business.printSchema()

In [0]:
import time

start = time.time()


def define_atraso(valor_atraso):
    if valor_atraso is None:
        return None
    elif valor_atraso > 15:
        return "Atrasado"
    elif valor_atraso == 0: 
        return "Pontual"
    else:
        return "Pequeno atraso"

udf_define_atraso = udf(define_atraso, StringType())

fat_voos_business_antecip_atrz = (fat_voos_business_antecip_atrz
    .withColumn("status_dep", udf_define_atraso("antec_dep"))
    .withColumn("status_arr", udf_define_atraso("antec_arr"))
)

fat_voos_business_antecip_atrz.count()

end = time.time()
print(f"Tempo de execução da célula: {end - start:.2f} segundos")

In [0]:
# Conta todas as linhas do DataFrame
num_linhas = fat_voos_business_antecip_atrz.count()
print(f"O DataFrame tem {num_linhas} linhas")


In [0]:
import time

start = time.time()

fat_voos_business_antecip_atrz2 = (
    fat_voos_business_antecip_atrz
    .withColumn("status_dep",
        when(col("antec_dep") < 0 , "atrasado")
        .when(col("antec_dep") == 0, "pontual")
        .when(col("antec_dep") > 0, "antecipado")
        .otherwise("Não informado"))
    .withColumn("status_arr",
        when(col("antec_arr") < 0, "antecipado")
        .when(col("antec_arr") == 0, "Pontual")
        .when(col("antec_arr") > 0, "atrasado")
        .otherwise("Não informado"))
)

fat_voos_business_antecip_atrz_2.count()

end = time.time()
print(f"Tempo de execução da célula: {end - start:.2f} segundos")


In [0]:
num_linhas = fat_voos_business_antecip_atrz2.count()
print(f"O DataFrame tem {num_linhas} linhas")

In [0]:
fat_voos_business_antecip_atrz2 = fat_voos_business_antecip_atrz2.select(
    col("data_voo"),
    col("sigla_icao_empresa_aerea"),
    col("empresa_aerea"),
    col("numero_voo"),
    col("iata_origem"),
    col("iata_destino"),
    col("partida_prevista"),
    col("partida_real"),
    col("chegada_prevista"),
    col("chegada_real"),
    col("antec_dep"),
    col("status_dep"),
    col("antec_arr"),
    col("status_arr")
).withColumn('load_data', expr("current_timestamp() - INTERVAL 3 HOURS"))

In [0]:
display(fat_voos_business_antecip_atrz2)

In [0]:
fat_voos_business_antecip_atrz2.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .partitionBy("sigla_icao_empresa_aerea") \
    .saveAsTable("`ct-esteira-dados-aviacao`.business.fat_voos_business_antecip_atrz")